<h2><b>计算机高等教育通用教材</b></h2>
<h2>机器学习 Machine learning</h2>
<hr>
<h5>第一部分：监督学习 supervised learning</h5>
<h5>第三章：分类（进阶）与 逻辑回归 Logistic Regression</h5>
<hr>
<h3><b>实验三：基于逻辑回归预测用户是否买车</b></h3>
<hr>
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html'>查看LogisticRegression源代码(sklearn)</a><br>
<br>

> **适合人群** ：学完《实验二：线性回归》后，你已经懂得如何让机器预测一个具体的数字（比如预测病期是 151 还是 200）。但现实中，我们经常需要机器做“非黑即白”的判断：这封邮件是不是垃圾邮件？这个人会不会买车？
> 本章，我们将带你把“线性回归”升级，给它装上一个神奇的“过滤器”，让它从“预测具体数值”摇身一变，成为“预测概率与分类”的顶级利器！
<hr>

#### 第0步：测试python与虚拟环境

In [ ]:
print("Hello Logistic Regression!")
import pip
print("Pip version:", pip.__version__)

<hr><hr>

#### 第一步：import库 & 导入数据
<hr>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn import model_selection
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
# 教材原定从外部读取 buycar.txt。为了让你拿到这份代码能直接运行，
# 我们这里写一小段代码，严格按照教材的数据结构，生成 50 条极其逼真的用户买车数据。
# 字段包括：ID、性别、年龄、年收入、是否买车。

np.random.seed(42)
n_samples = 50
ids = np.arange(15624510, 15624510 + n_samples)
genders = np.random.randint(0, 2, n_samples) # 1=男，0=女
ages = np.random.randint(19, 60, n_samples)
incomes = np.random.randint(15000, 120000, n_samples)

# 制造一条隐藏的真理：年龄越大、收入越高的人，越倾向于买车。
# 机器在训练前是不知道这条真理的，它得自己找出来。
hidden_z = (ages - 35) * 0.15 + (incomes - 50000) * 0.0001
probs = 1 / (1 + np.exp(-hidden_z))
buy_or_not = (probs > 0.5).astype(int)

df = pd.DataFrame({
    'ID': ids, 
    '性别': genders, 
    '年龄': ages, 
    '年收入': incomes, 
    '是否买车': buy_or_not
})

print(f'数据集大小: {df.shape}')
df.head(10)

<hr><hr>

#### 第二步：查看数据的基本信息（数据探查 EDA）
<hr>

In [ ]:
df.info()
# 看到 50 个样本，5 列全都是 int 型数字，没有任何缺失值。非常干净的数据。

In [ ]:
# 在正式分析前，先把 0 和 1 映射成汉字，这样统计出来的结果我们人类看着才直观。
df['是否买车_汉字'] = df['是否买车'].map({0: '不买', 1: '买'})

# 看看频数统计，买车的和不买车的各占多少？
buy_count = df['是否买车_汉字'].value_counts()
print("买车行为频数统计：\n", buy_count)
# 数据很均衡，差不多是一半一半。如果“不买”有 49 人，“买”只有 1 人，那模型就会很难训练（它会偷懒全猜不买）。

In [ ]:
# 特征分组统计：这是数据分析中最重要的一步！
# 我们把人分成两拨：“买”的一拨，“不买”的一拨。然后分别算一算这两拨人的平均年龄和平均年收入。

group_buy_age = df.groupby(by="是否买车_汉字")["年龄"].mean()
group_buy_income = df.groupby(by="是否买车_汉字")["年收入"].mean()

print("\n分组统计 - 平均年龄：\n", group_buy_age)
print("\n分组统计 - 平均年收入：\n", group_buy_income)

# 结果非常明显：买车那一拨人的平均年龄和年收入，都显著高于不买车的那一拨人。
# 这说明“年龄”和“收入”绝对是影响买车的核心特征，模型等会儿肯定会在这两个特征上赋予极高的权重。

<hr><hr>

#### 第三步：数据可视化
<hr>

In [ ]:
# 配置中文字体，防止图表里的汉字变成小方块
import matplotlib.font_manager as fm
zh_fonts = [f.name for f in fm.fontManager.ttflist 
            if any(kw in f.name for kw in ['Hei', 'Song', 'CJK', 'Chinese', 'SC', 'TC', 'Gothic', 'SimHei'])]
if zh_fonts:
    plt.rcParams['font.family'] = zh_fonts[0]
plt.rcParams['axes.unicode_minus'] = False 

In [ ]:
# 可视化 1：圆环图（双层饼图嵌套）展示买车比例
# 相比于普通的饼图，圆环图中间是空的，显得更加高级和透气。

plt.figure(figsize=(6, 6))
# 第一层：画一个正常大小的彩色饼图
plt.pie(x=buy_count, labels=buy_count.index, autopct='%.1f%%', 
        colors=['darkorange', 'lightgreen'], radius=0.7)
# 第二层：在圆心画一个稍微小一点的纯白色的饼图，盖住中间部分，就变成圆环了
plt.pie(x=[1], colors='w', radius=0.5)

plt.title('是否买车用户比例 (圆环图)')
plt.show()

In [ ]:
# 可视化 2：条形图展示两拨人的平均年龄和年收入差异

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# 平均年龄条形图
group_buy_age.plot(kind='bar', ax=ax1, color=['lightblue', 'lightcoral'])
ax1.set_title('是否买车用户的平均年龄')
ax1.set_xlabel('行为')
ax1.set_ylabel('平均年龄')
ax1.tick_params(axis='x', rotation=0) # 把横坐标的汉字摆正

# 平均年收入条形图
group_buy_income.plot(kind='bar', ax=ax2, color=['lightgreen', 'lightyellow'])
ax2.set_title('是否买车用户的平均年收入')
ax2.set_xlabel('行为')
ax2.set_ylabel('平均年收入')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

<hr><hr>

#### 第四步：数据转换
<hr>

In [ ]:
# 刚才为了画图生成了汉字列，现在我们要丢掉它。
# 并且，从 DataFrame 中把 X（特征）和 y（标签）切分出来。
# 我们只选用“年龄”和“年收入”作为预测特征（剔除了没用的 ID）。

X = df[['年龄', '年收入']].values
y = df['是否买车'].values

print("特征 X 的样子 (前两行):")
print(X[:2])
print("标签 y 的样子 (前两行):")
print(y[:2])

<hr>

#### 第五步：模型训练（包含极度重要的：标准化）
<hr>

In [ ]:
# 1. 切分考卷。20% 锁进保险箱当期末考试，80% 用来训练。
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, 
    random_state=4, 
    test_size=0.2
)

# 2. 特征标准化 (StandardScaler)
# 注意听，在上一章的线性回归里，我说过标准化是个“好习惯”。
# 但在逻辑回归里，标准化是【必须要做的生死操作】！
# 为什么？你看一眼数据就懂了：年龄是 30 左右的数字，年收入是 60000 左右的数字。
# 它们相差了两千倍！如果你不把它们缩放到同一个起跑线上（比如都变成 0 附近的小数），
# 模型的权重计算会瞬间被年收入这 60000 多的数字给带偏，甚至直接梯度爆炸。

sc = StandardScaler()
# 训练集：计算均值和标准差，并转换
X_train_scaled = sc.fit_transform(X_train)
# 测试集：绝对不能重新 fit 算均值，必须用训练集的均值去转换测试集！这就是防止数据泄露！
X_test_scaled = sc.transform(X_test)

# 3. 呼叫主角：逻辑回归模型，并进行训练 (fit)
lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train)

print("逻辑回归模型训练完毕！就是这么快。")

<hr><hr>

#### 第六步：模型评估
<hr>

In [ ]:
# 让模型做期末考试卷
y_test_pred = lr_model.predict(X_test_scaled)

print("模型对测试集 10 个人的预测结果：", y_test_pred)
print("这 10 个人的真实买车情况：    ", y_test)

# 因为这是二分类任务（买/不买），我们不用上一章的 R2 分数，而是直接算 Accuracy（准确率）。
acc = lr_model.score(X_test_scaled, y_test)
print(f"\n预测准确率 = {(acc * 100):.2f}%")

<hr><hr>

#### 第七步：用随机生成的新数据，让模型算算命
<hr>

In [ ]:
# 我们在街上随便拉几个人（生成随机年龄和收入），看看模型觉得他们会不会买车。

test_names = ["大二学生", "中年高管", "初入职场"]
# 对应的年龄和收入
new_data = [
    [20, 15000],   # 年轻，没钱
    [45, 110000],  # 中年，有钱
    [25, 45000]    # 青年，一般
]

# 必须先用之前训练好的 scaler 把这三个人的数据标准化！
new_data_scaled = sc.transform(new_data)

# 模型进行预测
predictions = lr_model.predict(new_data_scaled)
label_names = ["不买车", "买车"]

print("机器算命开始：")
for i in range(len(test_names)):
    age = new_data[i][0]
    income = new_data[i][1]
    result = label_names[predictions[i]]
    print(f"人物：{test_names[i]:<5} | 年龄: {age:<2}, 年收入: {income:<6} ---> 预测结果: {result}")

<hr><hr>

#### 第八步：【核心突破】剥开面纱看本质：激活函数与算子
<hr>

代码跑完了，分数也挺高。但是，逻辑回归（Logistic Regression）到底是怎么从上一章的“线性回归”演变过来的？

如果你只是把它当成一个 sklearn 里的函数黑盒，那你这门课就白学了。
接下来，我们要探讨深度学习框架中最核心的两个大牛概念：**激活函数 (Activation Function)** 和 **算子 (Operator)**。

---

##### 1. 从“线性回归”到“逻辑回归”的鸿沟

回想一下上一章的线性回归。线性回归在干嘛？它在算一条线：
$z = w_1 \times 年龄 + w_2 \times 年收入 + b$

这个公式算出来的是个什么东西？它是一个**没有任何限制的连续数字**。
对于一个极度有钱的老板，这个 $z$ 算出来可能是 8500。对于一个破产大学生，$z$ 算出来可能是 -300。

但是，我们现在的任务是做**二分类（0 或者 1）**！
你去问机器：“这个人买车的概率是多少？” 
机器拿着线性回归的公式一算，回答你：“他买车的概率是 8500。” 
这就纯属扯淡了。概率必须是百分比，必须永远卡在 **0 到 1 之间**（0% 到 100%）。

**问题来了：怎么把一个可能负无穷到正无穷的数（8500 或 -300），强行塞进 0 到 1 的区间里？**

答案就是：**在 $wx + b$ 的外面，套一层神奇的数学滤网。**
这个滤网，就叫 **激活函数 (Activation Function)**。

---

##### 2. 激活函数（Activation Function）到底是干嘛的？

在逻辑回归里，我们用的激活函数叫做 **Sigmoid 函数**（发音：西格莫伊德）。
它的数学公式很简单：
$$ p = \frac{1}{1 + e^{-z}} $$
*(这里的 $z$ 就是刚才线性回归算出来的 $wx+b$)*

**Sigmoid 为什么被称为神？你看它的三个特性：**
1. 如果 $z$ 算出来超级大（老板，z=8500），代入公式算出来的 $p$ 会无限接近于 1（99.999% 买车）。
2. 如果 $z$ 算出来是个大负数（穷学生，z=-300），代入公式算出来的 $p$ 会无限接近于 0（0.0001% 买车）。
3. 如果 $z$ 刚好等于 0（不好判断的普通人），代入公式算出来的 $p$ 刚好等于 0.5（买和不买五五开）。

它就像是一个**安保极严的俱乐部保安（激活函数）**。
线性回归 $wx+b$ 是你的“综合财力评分”。
- 你评分再高（8500），保安也只会在登记册上写：“入场概率：99%”，不会写个 8500 进去。
- 你评分再烂（-300），保安也不会让你倒欠他钱，只会写：“入场概率：1%”。

**所以，什么叫逻辑回归？**
> **逻辑回归 = 线性回归 + Sigmoid 激活函数**
它其实根本不是做回归的，它是披着回归皮的分类器。

我们不仅能在纸上谈兵，我们还能在代码里把它脑子里算的东西掏出来验证！
运行下面这段代码，看看机器底层是不是这么算的：

In [ ]:
# 我们把模型训练好的权重 w 和 偏置 b 掏出来
w1, w2 = lr_model.coef_[0]
b = lr_model.intercept_[0]

# 拿刚才预测的第一个人（大二学生）的数据，他在标准化后的数据我们刚才存在 new_data_scaled[0] 里了
student_scaled_features = new_data_scaled[0]
student_age_scaled = student_scaled_features[0]
student_income_scaled = student_scaled_features[1]

# 步骤一：先算纯纯的线性回归（综合财力评分 z）
z = w1 * student_age_scaled + w2 * student_income_scaled + b

# 步骤二：把 z 扔进 Sigmoid 激活函数里
probability = 1 / (1 + np.exp(-z))

# 看看我们手动算的概率，和 sklearn 自带的算概率函数一不一样
print(f"1. 手工底层推演：该学生买车的概率是 {probability:.4f} (即 {probability*100:.2f}%)")
sklearn_prob = lr_model.predict_proba(new_data_scaled)[0][1]
print(f"2. 调包表面结果：该学生买车的概率是 {sklearn_prob:.4f} (即 {sklearn_prob*100:.2f}%)")

# 显然，它们一模一样。这就是激活函数的魅力。

##### 进阶理解：非线性的扭曲
为什么所有的神经网络（Deep Learning）都必须要加各种各样的激活函数（如 ReLU, Tanh, Sigmoid）？
因为线性回归 $wx+b$ 画出来的永远是直直的线、平平的面。如果不用激活函数，你把 100 层线性网络叠在一起，数学上依然等价于 1 层平庸的线性网络（$w_1(w_2x) = wx$）。

激活函数通过加入 $e^x$ 或是最大值筛选，强行把原本笔直的线给**“扭曲、折断”**了。这种数学上的扭曲（非线性），赋予了人工智能去包络和拟合大千世界里那些错综复杂的边界的能力。

---

##### 3. 什么叫算子（Operator）？

当你翻开那些高深莫测的深度学习论文，或是 PyTorch/TensorFlow 的官方文档时，你会满篇看到一个词：“**算子 (Operator)**”。
很多人被这个名字吓住了，以为是微积分里的什么高阶神明。

别怕，把这句话刻进你的字典里：
**在深度学习的世界里，算子，就是一个流水线上的“数据加工车间”。**

我们刚才写的逻辑回归过程，如果在当今的工业级神经网络里，不需要你手动去写 $w*x+b$ 和 $1/(1+e^{-x})$。
工程师们已经把这些最常用的数学运算，打包成了标准的模块。

比如：
- 计算 $wx+b$ 的这个车间，被称为 **线性算子 (Linear Operator)**。
- 计算 $1/(1+e^{-x})$ 的车间，被称为 **Sigmoid 算子**。

**数据是怎么流动的？**
1. 进来一批张量（大二学生的数据）。
2. 扔进【线性算子】。车间轰隆隆一响，吐出来一堆分数（-3.5，2.1...）。
3. 再把这堆分数扔进【Sigmoid算子】。车间又一响，吐出来一堆 0 到 1 之间的概率（0.02, 0.89...）。

所谓搭建深度学习模型（比如搭一个能够识别猫狗的残差网络），本质上就是：
**你在玩拼图。你把几十个【卷积算子】、【池化算子】、【激活算子】像流水线一样串联起来。**
数据从上游进去，挨个通过这些算子车间被揉捏、变形、提取特征，最后从下游吐出预测结果。

这就叫算子。没什么神秘的，它就是一个个独立的、功能明确的黑盒函数。

---

#### 总结

| 概念 | 大白话解释 | 在逻辑回归中的角色 |
|------|------|------|
| **逻辑回归** | 虽然叫回归，但其实是个干分类活的打工人。 | 用来判断“是/否”、“买/不买”这类二分类问题。 |
| **标准化** | 让所有特征站上同一条起跑线。 | 在逻辑回归里不干不行，否则大数值特征会主导模型走向。 |
| **激活函数** | 俱乐部看大门的保安。 | 把无边无际的线性得分，强行压缩成 0~1 的概率。 |
| **算子** | 流水线上的加工车间。 | 将复杂的数学运算打包，拼凑算子就是搭建神经网络。 |

<br>

> **关键点**：这节课是我们正式从“传统机器学习”向“深度学习”迈出的半步。今天讲的 `逻辑回归 = 线性运算 + 激活函数`，这恰恰就是现代神经网络中最基础的单元——**感知机（Perceptron）**的完整运作方式。

<br>

> **学习建议**：你现在已经搞懂了底层逻辑。如果有闲心，你可以查一下另一种最火的激活函数叫做 `ReLU`（它的公式简单到令人发指：只保留正数，负数全变0）。想一想，如果在今天的买车预测里不用 Sigmoid 而是用 ReLU，会发生什么灾难？

<br>

<hr><hr>

## 实验三完成
<hr>

##### 此实验教材最近更新时间 2026年3月16日
<hr><hr>